# Use Gladius and Review Its Results Locally

## Goal and Learning Goals

Work through a small, completely offline practice run, then replace its inputs
with your own files. You will define a configuration, inspect parent/cutout
positions, review every accepted and flagged image, adjust quality screening
without recutting, and add display products using the same saved pixels.

The inputs below are **synthetic training images**, not Roman observations.
One exposure deliberately has an elongated central star, and a second target
crosses an image edge. Those defects make the review meaningful. This is not a
scientific calibration of the autocorrelation threshold.

## Setup

Use a kernel with the development version of `roman-gladius[notebook]` containing
named revisions and `gladius.inspection`; these additions are newer than PyPI
0.1.0a3. No MAST token or network access is required. Outputs stay under WORKSPACE.


### Choose the workspace

Use a new directory for the generated practice inputs. Re-running this notebook
reuses the saved synthetic inputs. `RUN_GUI` is optional so Run All can finish
without leaving a web server running.


In [ ]:
from pathlib import Path

WORKSPACE = Path.cwd() / "gladius-local-review"
RUN_GUI = False
NOTEBOOK_ORIGIN = "http://localhost:8888"  # Match your notebook's actual address.


### Create five practice exposures

This cell uses `create_demo` to establish the file/WCS format, then creates a
noisy multi-star field across five exposures in three filters. `demo2_cal.asdf` has the deliberately
elongated central star. The files and known defect are recorded here so you know
what to look for. Existing inputs are not overwritten on later notebook runs.


In [ ]:
import json
from copy import deepcopy
import asdf
import numpy as np
from astropy.time import Time
from gladius import RunConfig, Target, run
from gladius.core import pixel_to_world
from gladius.demo import create_demo

if not (WORKSPACE / "demo.yaml").exists():
    create_demo(WORKSPACE)
    rng = np.random.default_rng(27)
    yy, xx = np.indices((96, 96))
    bands = ["F062", "F184", "F213", "F062", "F213"]
    colours = {"F062": [1.4, .6, 1, .5, 1.4, .8],
               "F184": [1, 1, .8, 1.2, 1, 1.1],
               "F213": [.8, 1.4, .7, 1.5, .6, 1.2]}
    for index, band in enumerate(bands):
        template = WORKSPACE / "images" / f"demo{index % 2}_cal.asdf"
        with asdf.open(template, lazy_load=False, memmap=False) as af:
            tree = deepcopy(dict(af.tree))
        dx = 1.5 if index % 2 else 0.0
        data = rng.normal(0, 2, xx.shape)
        for star_number, (sx, sy, flux) in enumerate([(48,48,160),(20,20,85),(76,28,100),(28,74,70),(70,72,65),(48,12,50)]):
            flux *= colours[band][star_number]  # Illustrative colours, not a throughput simulation.
            sigma_x = 7 if index == 2 and sx == 48 else 2
            data += flux * np.exp(-0.5 * (((xx-sx-dx)/sigma_x)**2 + ((yy-sy)/2)**2))
        tree["roman"]["meta"]["instrument"]["optical_element"] = band
        tree["roman"]["data"] = data.astype("float32")
        tree["roman"]["err"] = np.full(xx.shape, 2, dtype="float32")
        tree["roman"]["var_poisson"] = np.maximum(data, 0).astype("float32")
        tree["roman"]["meta"]["exposure"]["start_time"] = Time(f"2026-01-0{index+1}T00:00:00")
        tree["roman"]["meta"]["exposure"]["end_time"] = Time(f"2026-01-0{index+1}T00:01:00")
        destination = WORKSPACE / "images" / f"demo{index}_cal.asdf"
        temporary = destination.with_suffix(".tmp.asdf")
        asdf.AsdfFile(tree).write_to(temporary)
        temporary.replace(destination)  # Only the synthetic files created above.
    (WORKSPACE / "training-inputs.json").write_text(json.dumps({
        "parent_count": 5, "deliberately_trailed_parent": "demo2_cal.asdf"}))
if not (WORKSPACE / "training-inputs.json").is_file():
    raise ValueError("Use a new WORKSPACE; this existing directory is not the practice field.")
print("Inputs:", sorted(p.name for p in (WORKSPACE / "images").glob("*_cal.asdf")))


## Understand the configuration before running

**SCI** is the science-pixel array, **ERR** its per-pixel uncertainty, and **DQ**
contains integer data-quality flags. **WCS** means the coordinate model mapping
pixels to sky positions. **ICRS** is the celestial reference frame used for the
RA/Dec coordinates here. ASDF and FITS are the saved file formats.

`RunConfig` is the Python form of the YAML. Omitted keys get defaults; misspelled
keys fail validation. Creating it and printing YAML do **not** query MAST.

| Section | What you choose | What this example does |
| --- | --- | --- |
| `run.name` | Name of the working project. | New settings become numbered revisions of this name. |
| `targets` | IDs and ICRS RA/Dec in **degrees**, or zero-based detector x/y. | Each selected parent is tried against every target. A catalogue position is a prediction, not a detected star. |
| `inputs` | Local directory/pattern or MAST criteria. | Explicit input set below; the query is printed before any request. |
| `selection` | UTC exposure-start dates `[start,end)`, detector numbers and filters. | Use dates/detectors/filters for your own sample; this exact-dataset example needs no date range. |
| `extraction` | Square side in pixels and boundary handling. | `pad` retains the requested size with unavailable pixels marked at detector edges; targets not on a parent detector are skipped. |
| `astrometry` | Supplied or corrected WCS; numerical checks. | Numerical round trips check internal consistency, **not** whether the image points at the right stars. |
| `quality` | Autocorrelation tile, lag radius and threshold. | A provisional elongation screen; a failed screen still saves the native cutout. |
| `stacks` | Independent median, GIF and RGB choices. | Display products use eligible cutouts; native images are for measurements. |
| `storage` | Output location, ASDF/FITS and parent retention. | Retain parents for context images; local original files are never deleted. |
| `execution` | Reuse, continue after errors, lock timeout. | Reuse enabled; stop on errors. Inspect manifests even if a run returns. |

Use the GUI below to inspect help and build YAML, or the
[full configuration reference](https://roman-gladius.readthedocs.io/en/latest/configuration.html).
Changing dates/queries refreshes discovery. Changing quality or display settings
reuses verified native cutouts. `run(config, refresh_inputs=True)` refreshes MAST
membership; `fresh=True` deliberately bypasses reuse. Completed revisions stay intact.


In [ ]:
from gladius.io import read_exposure

parent = read_exposure(WORKSPACE / "images/demo0_cal.asdf")
ra, dec = pixel_to_world(parent.wcs, 20, 20)
config = RunConfig(
    run={"name": "practice-field"},
    targets=[Target(id="central-star", ra=270, dec=-29),
             Target(id="edge-star", ra=float(ra), dec=float(dec))],
    inputs={"mode": "directory", "directory": "images", "pattern": "*_cal.asdf"},
    selection={"start": "2026-01-01", "end": "2026-01-06",
               "detectors": [1], "filters": ["F062", "F184", "F213"]},
    extraction={"size": 64, "edge": "pad"},
    quality={"autocorrelation_limit": 2.6, "tile_size": 48, "radius": 10,
             "exclude_from_stacks": True},
    stacks={"enabled": False, "gif_enabled": False},
    storage={"output_directory": "output", "retain_full_images": False,
             "formats": ["asdf", "fits"]},
).resolved(WORKSPACE)
del parent


### Read and save the full YAML

The local pattern is limited to this workspace's `images` directory. No archive
query exists in this configuration. For your own local images, change the directory,
pattern, targets and selections; do not modify this synthetic input generator.


In [ ]:
import yaml

print(yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))
(WORKSPACE / "practice-config.yaml").write_text(
    yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))


## Make the native cutouts

Five input images × two targets gives ten attempted pairs. The edge target has
padding; padding is retained and flagged, not evidence of missing catalogue data.
The progress log identifies extraction versus reuse.


In [ ]:
manifest_path = run(config, progress=print)
manifest = json.loads(manifest_path.read_text())
print("Manifest:", manifest_path)
print("Work performed:", manifest["stage_counts"])


## Check every attempted parent/target pair

The audit table includes **complete, skipped and failed** entries. `accepted=False`
means a complete native cutout was flagged by the optional screen; it does not
mean the cutout was discarded. `target_off_detector` is different: the target is
not on that detector. Errors are failures, not quality judgements.

Inspect the table before trusting the previews. If you expected more parent images,
compare the input list/query and date/filter selection; quality screening does not
remove a parent from discovery. A complete run does not prove a star is centred
or that every catalogue target was on every detector.


In [ ]:
from astropy.table import Table
from IPython.display import display
from gladius.inspection import audit_rows, contact_sheets, parent_preview

# A Python list of dictionaries: one row for every attempted input/target pair.
review = audit_rows(manifest_path)
audit = Table(rows=review)
display(audit[["target", "parent", "filter", "status", "accepted", "display_eligible", "axis_ratio", "reason"]])
display(audit[["target", "parent", "padding_fraction", "do_not_use_fraction", "saturated_fraction"]])
print("Run status:", manifest["status"])
print("Input images used:", len(manifest["sources"]))
print("Parents rejected by date/detector/filter:", manifest["selection_rejections"])
print("Recorded errors:", manifest["errors"])


## What was cut out, and is it the right place?

Each parent preview has **requested cutout boxes** and **predicted target crosses**.
It shows the original detector orientation; N/E arrows come from the saved WCS.
Boxes may cross an edge when padding is allowed. Purple areas in cutout previews
are unavailable pixels, not empty sky.

Compare the star pattern and target location against a finding chart. The cross
is not a fitted centroid. An internally consistent but incorrect WCS can put it
on the wrong star. An independent finding chart or known reference sources are
needed to test absolute pointing; a parent and its own cutout are not independent.

The optional reference-image cell below makes a side-by-side visual comparison
possible by displaying your chart in the same notebook. It does not rotate,
register, cross-match, or automatically certify a match. Match scale/orientation
using the chart's labels and these N/E arrows before judging the star pattern.


In [ ]:
# Show every retained/available parent, once, with all its completed cutout boxes.
parent_ids = sorted({entry["parent_sha256"] for entry in manifest["entries"].values()
                     if entry["status"] == "complete"})
for parent_id in parent_ids:
    display(parent_preview(manifest_path, parent_id, width=800))


In [ ]:
from PIL import Image

FINDING_CHART = None
# FINDING_CHART = Path("/path/to/finding-chart.png")  # Optional independent chart.
if FINDING_CHART is not None:
    with Image.open(FINDING_CHART) as chart:
        reference = chart.convert("RGB")
        reference.thumbnail((900, 900))
        display(reference)


## Review ALL cutouts, including the flagged ones

The following pages include every complete cutout; none are hidden because the
screen rejected them. Compare **KEEP** and **FLAG** thumbnails to look for both
false exclusions and missed defects. The table above retains every full filename.

- A cross marks the requested position. N/E arrows retain the native orientation.
- The gold rectangle is the central region used by autocorrelation. Damage outside
  it can be missed: the screen is not a whole-image quality classifier.
- The score is an autocorrelation long/short axis ratio: near 1 is roughly round;
  higher is more elongated. It is not a fitted stellar PSF shape.
- The same target/filter/unit group shares one asinh brightness stretch, so changes
  are visible. Different groups have their own stretch; displayed brightness is not
  a calibrated cross-filter comparison.
- Padding and bad/saturated pixel fractions help identify missing coverage or DQ
  problems. Inspect artifacts, centring and background by eye too.

`insufficient_tile`, `insufficient_signal` and `degenerate_acf` are failed
measurements, **not** evidence of trailing. A smaller cutout than `tile_size` will
always fail the enabled screen. Do not solve that by blindly raising its threshold.


In [ ]:
# Twelve images per PNG page; iterate every page to review all complete cutouts.
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    path = review_directory / f"all-cutouts-{page_number:03d}.png"
    sheet.save(path)
    display(sheet)
print("Review PNGs:", review_directory)


## Tune the screen without downloading or cutting again

Use the audit and thumbnails to choose a threshold. A value such as **2.6** is an
example, not a calibrated recommendation for Roman. Increase it to tolerate more
elongation; decrease it to flag more images. `None` disables the screen. A score
**at or above** the threshold is flagged. `tile_size` is the side length, in pixels, of the central square used for the
screen. `radius` is the largest pixel shift included when measuring the shape of
that square's autocorrelation (its similarity to shifted copies of itself).
These choices also affect the score; changing them measures a different region/shape and needs visual checking.

When `exclude_from_stacks=True`, flagged cutouts are omitted from display products.
They remain available for inspection and scientific decisions. Setting it to False
includes them in displays; it does not clear their flags. There is currently no
per-exposure manual keep/exclude override in the tool: use these settings and
explicit input/date selections, and retain your reasons alongside the run.

The next cell creates a revision of the same named run. With the native cache
available, it does not repeat extraction. The revision manifest has the **current**
quality decision; reused image headers retain their original creation provenance.


In [ ]:
# Edit these values after inspecting the first audit and contact sheets.
NEW_ACF_LIMIT = 2.0  # Demonstration only; choose based on representative images.
NEW_TILE_SIDE = config.quality.tile_size
NEW_LAG_RADIUS = config.quality.radius

settings = config.model_dump(mode="json")
settings["quality"].update(autocorrelation_limit=NEW_ACF_LIMIT,
                           tile_size=NEW_TILE_SIDE, radius=NEW_LAG_RADIUS)
review_config = RunConfig.model_validate(settings)
old_manifest_path = manifest_path
manifest_path = run(review_config, progress=print)
manifest = json.loads(manifest_path.read_text())
print("Previous revision:", old_manifest_path.parent)
print("Current revision:", manifest_path.parent)
print("Work performed:", manifest["stage_counts"])
# With the cache intact, 'extracted' is 0. Original revision files remain unchanged.
audit = Table(rows=audit_rows(manifest_path))
display(audit[["target", "parent", "filter", "accepted", "display_eligible", "axis_ratio", "reason"]])
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    sheet.save(review_directory / f"all-cutouts-{page_number:03d}.png")
    display(sheet)
config = review_config


## Add or rebuild display products without repeating extraction

A **median stack** combines aligned exposure pixels. A **GIF** plays individual
aligned exposures in time order. An **RGB composite** maps three independently
normalized filter medians to colours. They are different products and can be
selected independently. A one-exposure median is just that exposure; it does not
provide a noise-reduction demonstration.

Adding a composite reuses native cutouts and existing independent previews.
RGB needs three eligible filters for the same target; the example below prints
missing-filter reasons instead of pretending that one filter is a colour image.
For a real composite, select actual blue/green/red filters present in your input
inventory and use a bounded dataset/program/date search that includes them.

Stacks/GIFs/composites are for inspection. They resample pixels and do not propagate
uncertainties or perform PSF/photometric matching. Use native cutouts for flux or
position measurements. Check the display manifest's exact contributors and exclusions.

The local practice field has F062, F184 and F213 inputs, so it makes actual RGB
previews as well as per-filter stacks/GIFs. Its colours are illustrative synthetic
flux weights, not a Roman throughput or source-spectrum simulation.


In [ ]:
settings = config.model_dump(mode="json")
settings["stacks"].update(enabled=True, gif_enabled=True, rgb_enabled=True,
                          blue_filter="F062", green_filter="F184", red_filter="F213")
config = RunConfig.model_validate(settings)
manifest_path = run(config, progress=print)
manifest = json.loads(manifest_path.read_text())
display_manifest = json.loads((manifest_path.parent / "stack-manifest.json").read_text())
print("Native work:", manifest["stage_counts"])
for category in ("products", "animations", "composites", "composite_skips", "excluded"):
    print(category, display_manifest[category])


In [ ]:
from astropy.io import fits
from IPython.display import Image as DisplayImage
from gladius.inspection import image_preview

# Median images: display stretch only; the FITS science values stay unchanged.
for product in display_manifest["products"]:
    path = manifest_path.parent / product["path"]
    with fits.open(path) as hdul:
        print("Median:", product["target"], product["filter"], "inputs:", len(product["inputs"]))
        display(image_preview(hdul[0].data))
        coverage = hdul["COVERAGE"].data
        print("Contributing exposures per output pixel:", int(coverage.min()), "to", int(coverage.max()))

# The GIF is individual aligned exposures, NOT the median shown above.
for animation in display_manifest["animations"]:
    print("GIF:", animation["target"], animation["filter"], "frames:", len(animation["frames"]))
    display(DisplayImage(filename=str(manifest_path.parent / animation["path"])))

# RGB has independently stretched filter channels; it is not physical colour.
for composite in display_manifest["composites"]:
    print("RGB:", composite["target"], composite["channel_filters"])
    with Image.open(manifest_path.parent / composite["path"]) as image:
        display(image.resize((300, 300), Image.Resampling.NEAREST))


## Build and edit the same configuration in the GUI

Set `RUN_GUI=True` in the setup cell, then run the next cell. It loads the current
reviewed configuration and provides field help, YAML import/export and **Make**.
You can use it to make YAML without starting a run. The GUI runs in this kernel.

For local JupyterLab, set NOTEBOOK_ORIGIN to the scheme/host/port in the notebook address, without its path.
Changing settings in the GUI does not automatically change the Python `config`
variable: download its YAML and load it with `load_config(path)` to use it in code.
Stop the server below before restarting it with a different configuration.


In [ ]:
if RUN_GUI:
    import threading
    from IPython.display import IFrame, display
    from gladius.web import GuiServer

    if "gui_thread" not in globals() or not gui_thread.is_alive():
        gui_server = GuiServer(WORKSPACE, port=0, iframe_origin=NOTEBOOK_ORIGIN, initial_config=config)
        gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
        gui_thread.start()
    gladius_ui_url = gui_server.url + "/"
    print("Open the GUI in a separate tab:", gladius_ui_url)
    display(IFrame(src=gladius_ui_url, width="100%", height=650))
else:
    print("GUI skipped. Set RUN_GUI=True in the settings cell to explore it.")


### Stop the GUI when finished

Wait for any job to finish first. Closing an iframe does not stop the server.


In [ ]:
if "gui_thread" in globals() and gui_thread.is_alive():
    gui_server.shutdown()
    gui_server.server_close()
    gui_thread.join(timeout=5)
    print("Embedded GUI stopped.")


## Next Steps

For your own local data, replace `targets` with `read_targets_csv(path)` and point
`inputs.directory` at calibrated SCI/ERR/DQ images. Use explicit UTC dates,
detectors and filters if that directory contains more than your desired sample.
For archive data, use the separate Nexus notebook's bounded-query workflow.

You have inspected parents, every completed cutout, skips/failures and quality
flags; changed screening without extraction; and requested independent displays.
The synthetic known defect lets you check whether a threshold behaves as expected.
It does not establish a threshold for real observations.

Authors: Gladius contributors. Updated 2026-11-02. Sources: Gladius API/configuration,
Astropy coordinate/WCS tools, ASDF and Pillow. 
